# 41 モデル構築

| 項目 | 内容 |
|------|------|
| プロジェクトID | BAXX-XXX-XX |
| タスク種別 | 回帰 / 分類 |
| 入力データ | `processed_data/` |
| 出力 | `results/tables/model.pkl`, 評価レポート |
| 作成者 | |
| 更新日 | YYYY-MM-DD |

## 目的・方針
<!-- 予測したいものと、使用するアルゴリズムの選定理由を記述 -->

## 特徴量リストと選定理由
<!-- どの変数を使い、なぜ使うかを事前に記述する（後からでも可） -->
| 特徴量 | 説明 | 採用理由 |
|--------|------|----------|
| | | |

---
## 0. 環境セットアップ

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib

sys.path.append(str(Path('../../').resolve()))
from funcs import data_loader, preprocessing, feature_engineering, model, visualization

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

In [ ]:
cfg      = data_loader.load_yaml('../../configs/configs.yaml')
viz_cfg  = visualization.setup_matplotlib('../../configs/visualize.yaml', theme='notebook')

RANDOM_SEED   = cfg.get('random_seed', 42)
PROCESSED_DIR = Path(cfg['paths']['processed_data'])
FIGURES_DIR   = Path(cfg['paths']['figures'])
TABLES_DIR    = Path(cfg['paths']['tables'])
np.random.seed(RANDOM_SEED)

---
## 1. データ読み込み

In [ ]:
# df = data_loader.load_parquet(PROCESSED_DIR / 'processed.parquet')
# print(df.shape)
# df.head(3)

---
## 2. 特徴量エンジニアリング

In [ ]:
# カレンダー特徴量
# df = feature_engineering.add_calendar_features(df, date_col='date')
# df = feature_engineering.add_special_day_flags(df, date_col='date')

# ラグ特徴量
# df = feature_engineering.add_lag_features(
#     df, cols=['value'], lags=[1, 3, 6], group_col='id', sort_col='date'
# )

# ローリング統計量
# df = feature_engineering.add_rolling_features(
#     df, cols=['value'], windows=[3, 6], agg_funcs=['mean', 'std'],
#     group_col='id', sort_col='date'
# )

# ラグ・ローリング特徴量を作ったら欠損が生じるため確認
# print(df.isnull().sum()[df.isnull().sum() > 0])
# df = df.dropna().reset_index(drop=True)

---
## 3. 学習/テスト分割と特徴量定義

In [ ]:
# 時系列分割
# SPLIT_DATE = cfg['period']['train_end']  # configs.yamlから取得
# train, test = preprocessing.train_test_split_by_date(df, date_col='date', split_date=SPLIT_DATE)

In [ ]:
# 目的変数と特徴量の定義
# TARGET = 'target_col'
# DROP_COLS = ['date', 'id', TARGET]  # 使わない列
# FEATURES  = [c for c in train.columns if c not in DROP_COLS]

# print(f'特徴量数: {len(FEATURES)}')
# print('特徴量一覧:', FEATURES)

# X_train, y_train = train[FEATURES], train[TARGET]
# X_test,  y_test  = test[FEATURES],  test[TARGET]

---
## 4. モデルパラメータ設定

In [ ]:
# LightGBM パラメータ（回帰の場合）
# lgbm_params = {
#     'objective':        'regression',
#     'n_estimators':     500,
#     'max_depth':        7,
#     'num_leaves':       31,
#     'learning_rate':    0.05,
#     'colsample_bytree': 0.8,
#     'subsample':        0.8,
#     'reg_alpha':        0.1,
#     'reg_lambda':       0.1,
#     'random_state':     RANDOM_SEED,
#     'n_jobs':           -1,
# }

# LightGBM パラメータ（分類の場合）
# lgbm_params = {
#     'objective':        'binary',
#     'metric':           'auc',
#     'n_estimators':     500,
#     'max_depth':        7,
#     'num_leaves':       16,
#     'learning_rate':    0.005,
#     'colsample_bytree': 0.4,
#     'subsample':        0.6,
#     'random_state':     RANDOM_SEED,
#     'n_jobs':           -1,
# }

---
## 5. モデル学習（クロスバリデーション）

In [ ]:
# TASK = 'regression'  # 'regression' or 'binary'
# N_SPLITS = 5

# print('=== クロスバリデーション ===')
# models, fold_metrics, oof_pred = model.build_lgbm_cv(
#     X_train, y_train,
#     params=lgbm_params,
#     n_splits=N_SPLITS,
#     task=TASK,
#     random_state=RANDOM_SEED
# )

---
## 6. 特徴量重要度

In [ ]:
# 全foldの平均重要度
# import pandas as pd
# imp_list = [model.get_feature_importance(m, FEATURES) for m in models]
# imp_avg  = pd.concat(imp_list).groupby('feature')['importance'].mean().reset_index()
# imp_avg  = imp_avg.sort_values('importance', ascending=False)

# fig = visualization.plot_feature_importance(imp_avg, top_n=25)
# visualization.save_figure(fig, FIGURES_DIR, '41_feature_importance.png')
# imp_avg.head(20)

---
## 7. テストデータ評価

In [ ]:
# アンサンブル予測（全foldの平均）
# if TASK == 'regression':
#     y_pred_test = np.mean([m.predict(X_test) for m in models], axis=0)
#     metrics = model.calc_regression_metrics(y_test, y_pred_test)
# else:
#     y_prob_test = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)
#     y_pred_test = (y_prob_test >= 0.5).astype(int)
#     metrics = model.calc_classification_metrics(y_test, y_pred_test, y_prob_test)

# model.print_metrics(metrics, title='テストデータ評価')

In [ ]:
# 実績 vs 予測（回帰）
# fig = visualization.plot_actual_vs_predicted(y_test, y_pred_test)
# visualization.save_figure(fig, FIGURES_DIR, '41_actual_vs_predicted.png')

# CAP曲線（分類）
# fig, ax = plt.subplots(figsize=(7, 6))
# model.plot_cap_curve(y_test, y_prob_test, ax=ax)
# visualization.save_figure(fig, FIGURES_DIR, '41_cap_curve.png')

---
## 8. モデル・結果の保存

In [ ]:
# モデル保存
# model.save_model(models, TABLES_DIR / 'lgbm_models.pkl')

# 特徴量リスト保存
# data_loader.save_csv(
#     pd.DataFrame({'feature': FEATURES}),
#     TABLES_DIR / 'features.csv'
# )

# OOF予測値の保存
# oof_df = train[['date', 'id', TARGET]].copy()
# oof_df['oof_pred'] = oof_pred
# data_loader.save_csv(oof_df, TABLES_DIR / 'oof_predictions.csv')